# Fine-tune DARE2D (transfer learning)

Companion to `Run_dare2d_Prediction.ipynb` (runs the model) and `Run_dare2d_Retraining.ipynb`
(retrains **from scratch**). This notebook **fine-tunes** a pretrained DARE2D checkpoint: it loads
the weights into the matching architecture, **freezes the backbone**, and continues training at a
low (optionally discriminative) learning rate -- so you adapt an existing model to new data instead
of training from zero.

### Command-line twin of the napari plugin
This notebook is the command-line twin of the **"Transfer learning / fine-tune"** mode in the
DARE2D napari retraining widget (`napari-dare2d/napari_dare2d/_widget.py::retrain_widget`). Both
front-ends launch the **same engine** -- `training/torch/finetune.py` -- as a subprocess. The widget
exposes the knobs as controls; this notebook exposes them as variables. Anything you do here you can
do with the widget, and vice-versa.

> Fine-tuning is **PyTorch-only** by design (it is the lead path). The base model must be a `.pt`;
> the curated demo models ship `.pt` beside each `.h5`, and you can convert your own `.h5` with
> `dare2d-torch/convert_to_torch.py`.

## How each setting maps to the napari widget

| Notebook variable | napari widget control (fine-tune mode) | `finetune.py` flag |
|---|---|---|
| `base_model` | **Load model to fine-tune (.pt)** | `--base-model` |
| `stage` | **Model** (Regression / Segmentation -- one stage per `.pt`) | `--experiment` |
| `test_set` / `train_sets` | **Test set (held out)** / **Train sets** (leave-one-out) | `--test-set` / `--train-sets` |
| `run_name` | **Run name** | `--run-name` |
| `unfreeze_last` | **Unfreeze last N blocks** | `--unfreeze-last` |
| `bn_mode` | **Frozen-backbone BN** (frozen / adapt) | `--bn-mode` |
| `ft_lr` | **Fine-tune LR** | `--ft-lr` |
| `discriminative` / `backbone_lr_mult` | **Discriminative LR** / **Backbone LR x** | `--no-discriminative` / `--backbone-lr-mult` |
| `weight_decay` | **Weight decay** | `--weight-decay` |
| `lr_schedule` / `warmup_epochs` | **LR schedule** / **Warmup epochs** | `--lr-schedule` / `--warmup-epochs` |
| `grad_clip` | **Grad clip (max-norm)** | `--grad-clip` |
| `augment` / `augment_strength` | **Augmentation** / **Augment strength** | `--no-augment` / `--augment-strength` |
| `patience` | **Early-stop patience** | `--patience` |
| `epochs` / `steps` / `batch_size` | same-named controls | `--epochs` / `--steps` / `--batch-size` |

Everything outside the widget's **Advanced parameters** section runs on the same sensible defaults
you see below.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

# Resolve the repo root robustly (honour DARE2D_BASE_DIR, else walk up to the folder with setup.py)
# -- same pattern as the prediction/retraining notebooks.
def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in (start, *start.parents):
        if (cand / "setup.py").exists():
            return cand
    return start

base_dir = Path(os.environ.get("DARE2D_BASE_DIR", str(_find_repo_root(Path.cwd()))))
os.chdir(base_dir)                       # finetune.py is launched repo-root-relative, like the widget
finetune_py = base_dir / "training" / "torch" / "finetune.py"
assert finetune_py.exists(), f"finetune.py not found under {base_dir}; run from the DARE2d repo root."
print("Base dir  :", base_dir)
print("Engine    :", finetune_py)
print("           (this is exactly what the widget's 'Start retraining' runs in fine-tune mode)")

## 1) Base model + stage + data

`base_model` = the widget's **Load model to fine-tune** button; `stage` = the **Model** selector
(one stage per `.pt`); `test_set` / `train_sets` = the **Test set** / **Train sets** controls
(leave-one-out -- val is the held-out test set, exactly as in scratch retraining).

In [ ]:
stage       = "regression"               # "regression" or "segmentation"
experiment  = {"regression": "regression2d", "segmentation": "segmentation2d"}[stage]

# Any DARE2D .pt for that stage. Default = the curated demo checkpoint for set 8.
base_model  = (base_dir / "models" / "demo" / "neuroepithelium"
               / f"{stage}_checkpoints" / "checkpoints_set_8_all_but_target" / "best.pt")

test_set    = 8                          # held-out validation set
train_sets  = ""                         # comma list (e.g. "1,2,3"); blank = all other sets

assert base_model.exists(), (
    f"base model not found: {base_model}\n"
    "Fetch the demo data (napari 'Download DARE2D data'), or point base_model at your own .pt "
    "(convert a .h5 with dare2d-torch/convert_to_torch.py).")
print(f"stage      = {stage}  ({experiment})")
print(f"base_model = {base_model}")
print(f"test_set   = {test_set}   train_sets = {train_sets or '(all but test)'}")

## 2) Fine-tuning parameters

These are the widget's **Advanced parameters** (collapsed by default there). The defaults match the
widget's defaults; change only what you need.

- **Freeze depth** (`unfreeze_last`) -- the backbone is frozen; unfreeze the last *N* backbone blocks
  (0 = head only). Keras `layer.trainable` / PyTorch `requires_grad`, here via PyTorch.
- **Frozen-backbone BN** (`bn_mode`) -- `"frozen"` (default) holds the frozen backbone's BatchNorm
  running stats fixed (BN `.eval()`, a true freeze); `"adapt"` lets them **re-estimate** on the new
  data while the weights (incl. BN gamma/beta) stay frozen -- for a larger, distribution-shifted set.
- **Discriminative LR** -- a lower LR for the unfrozen backbone (`ft_lr * backbone_lr_mult`) than the head.
- **Schedule** -- linear **warmup** then **cosine** decay (or constant), plus optional gradient clipping.
- **Early stopping** + best-checkpoint saving, **augmentation** toggle/strength.

In [ ]:
run_name         = "finetune_demo"       # output -> models/<run_name>/...  (never overwrites curated demo models)
epochs           = 20
steps            = 500                    # optimizer steps per epoch
batch_size       = 32

unfreeze_last    = 0          # unfreeze the last N backbone blocks (0 = head only)
bn_mode          = "frozen"   # frozen-backbone BatchNorm: "frozen" (stats fixed) | "adapt" (re-estimate on new data)
ft_lr            = "1e-4"     # fine-tune LR for the (unfrozen) head -- lower than scratch
discriminative   = True       # lower LR for the unfrozen backbone than the head
backbone_lr_mult = "0.1"      # unfrozen-backbone LR = ft_lr * this
weight_decay     = "1e-4"     # AdamW weight decay
lr_schedule      = "cosine"   # "cosine" (warmup -> cosine) or "constant"
warmup_epochs    = 1
grad_clip        = "1.0"      # gradient max-norm; "0" = off
augment          = True       # apply the training-data augmentations
augment_strength = "1.0"      # scales augmentation probabilities (0..1)
patience         = 0          # early-stop after N epochs w/o val improvement; 0 = off
seed             = 12345

## 3) Run fine-tuning

The cell below builds the **exact** `python training/torch/finetune.py ...` command the widget's
**Start retraining** button launches in fine-tune mode (mirrors `_cmd` in `_widget.py`), then streams
its output -- the same `[phase]` / `[step]` / `[epoch]` progress markers the widget's progress bar reads.
A GPU is strongly recommended. Interrupt the kernel to stop.

In [ ]:
# Mirror of napari-dare2d/napari_dare2d/_widget.py::retrain_widget._cmd (fine-tune branch)
cmd = [sys.executable, str(finetune_py),
       "--experiment", experiment, "--base-model", str(base_model),
       "--test-set", str(test_set), "--run-name", run_name,
       "--epochs", str(epochs), "--steps", str(steps), "--crop", "256",
       "--batch-size", str(batch_size),
       "--unfreeze-last", str(unfreeze_last), "--bn-mode", bn_mode, "--ft-lr", ft_lr,
       "--backbone-lr-mult", backbone_lr_mult, "--weight-decay", weight_decay,
       "--lr-schedule", lr_schedule, "--warmup-epochs", str(warmup_epochs),
       "--grad-clip", grad_clip, "--augment-strength", augment_strength,
       "--patience", str(patience), "--seed", str(seed)]
if train_sets.strip():
    cmd += ["--train-sets", train_sets.strip()]
if not discriminative:
    cmd += ["--no-discriminative"]
if not augment:
    cmd += ["--no-augment"]


def run(cmd):
    '''Stream the fine-tune subprocess (same mechanism as the widget's worker) and capture the
    saved checkpoint path from the final '[finetune] DONE ... checkpoint:' line.'''
    print("> " + " ".join(cmd) + "\n")
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, text=True, env=env)
    ckpt = None
    for line in proc.stdout:
        sys.stdout.write(line)
        if "checkpoint:" in line:
            ckpt = line.split("checkpoint:", 1)[1].strip()
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"fine-tune exited with code {proc.returncode}")
    return ckpt


ckpt_path = run(cmd)
print("\nFine-tuned checkpoint:", ckpt_path)

## 4) Round-trip: the result loads straight into inference

The fine-tuned `best.pt` is loaded here with `torch_backend.load_torch_regression` /
`load_torch_segmentation` -- the **same loader the inference widget uses** (`_api._pytorch_builder`).
So a checkpoint that loads here drops directly into the napari **DARE2D division detection** widget,
no conversion needed.

In [ ]:
import numpy as np
sys.path.insert(0, str(base_dir / "dare2d-torch"))
import torch_backend as tb

if stage == "regression":
    pred = tb.load_torch_regression(ckpt_path)
    length, angle = pred.model.predict(np.random.rand(2, 64, 64, 3).astype("float32"), verbose=0)
    print("OK - fine-tuned regression model loads via the inference backend; "
          f"length {np.asarray(length).shape}, angle {np.asarray(angle).shape}")
else:
    pred = tb.load_torch_segmentation(ckpt_path)
    out = pred.model.predict(np.random.rand(1, 256, 256, 3).astype("float32"), verbose=0)
    print(f"OK - fine-tuned segmentation model loads via the inference backend; output {np.asarray(out).shape}")

## 5) The config sidecar

Every fine-tune writes a `finetune_config.json` next to `best.pt` (the widget does this too): the
seed, **all** hyperparameters, and a sha256/name/size of the **base model** you fine-tuned from -- so a
run is reproducible and traceable back to its parent.

In [ ]:
sidecar = Path(ckpt_path).parent / "finetune_config.json"
print(json.dumps(json.loads(sidecar.read_text(encoding="utf-8")), indent=2))

## 6) Use your fine-tuned model

Point inference at the new checkpoint exactly as you would a curated one:

- **napari plugin** -- open **DARE2D division detection** and set the **Regression / Segmentation
  checkpoint** field to the `..._checkpoints` folder holding this `best.pt`.
- **Prediction notebook** -- set `reg_dir` / `seg_dir` to that folder.

Because the fine-tuned `best.pt` is byte-compatible with the inference backend (verified in step 4),
there is nothing else to do.